In [1]:
# 标准库
import os
import sys
import json
import re
from pathlib import Path
from glob import glob

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
def parse_config_name(config_str):
    """
    把类似 'calib-arc_challenge_prompt-cot_method-WIFV_strategy-logistic_ratio-0.2'
    解析成 {'calib': 'arc_challenge', 'prompt': 'cot', 'method': 'WIFV', 'strategy': 'logistic', 'ratio': '0.2'}
    """
    parts = config_str.split("_")
    config_dict = {}
    current_key = None
    for part in parts:
        if "-" in part:
            key, val = part.split("-", 1)
            config_dict[key] = val
            current_key = key
        elif current_key:
            config_dict[current_key] += "_" + part  # 把长值合并
    return config_dict



In [3]:
def lm_eval_results_to_df(src) -> pd.DataFrame:
    if isinstance(src, (str, Path)):
        data = json.loads(Path(src).read_text())
    elif isinstance(src, dict):
        data = src
    else:
        raise TypeError("src 必须是文件路径或已解析 dict")

    rows = []
    order = {t: i for i, t in enumerate(data["results"].keys())}
    for task, res in data["results"].items():
        ver  = data.get("versions", {}).get(task, "")
        shot = data.get("n-shot", {}).get(task, "")
        hib  = data.get("higher_is_better", {}).get(task, {})
        for key, val in res.items():
            if "_stderr" in key or "," not in key:
                continue
            metric, flt = key.split(",", 1)
            stderr_key  = f"{metric}_stderr,{flt}"
            stderr_val  = res.get(stderr_key, None)

            # 构造 "value ± stderr" 字符串
            if stderr_val is not None:
                value_str = f"{val:.4f} ± {stderr_val:.4f}"
            else:
                value_str = f"{val:.4f}"

            rows.append(
                dict(
                    Task=task,
                    Version=ver,
                    Filter=flt,
                    Shot=shot,
                    Metric=metric,
                    Arrow="↑" if hib.get(metric, True) else "↓",
                    Value=val,
                    Stderr=stderr_val,
                    Display=value_str,
                    _order=order[task],
                )
            )

    return (
        pd.DataFrame(rows)
          .sort_values(["_order", "Metric"])
          .drop(columns="_order")
          .reset_index(drop=True)
    )



In [4]:
def collect_eval_results(root_dir, pattern="**/.__tmp__*/results_*.json"):
    """
    扫描 root_dir 下所有 lm-eval 结果文件，自动提取 Model 和 Config 属性列。
    按 Task ↑，Metric ↓，Value ↑ 排序。
    """
    root_dir = Path(root_dir)
    files = sorted(glob(str(root_dir / pattern), recursive=True))
    if not files:
        raise FileNotFoundError(f"未在 {root_dir} 中找到符合 {pattern} 的文件")

    all_frames = []
    for fp in files:
        df = lm_eval_results_to_df(fp)
        parts = Path(fp).parts
        try:
            model_name  = parts[-4]
            config_name = parts[-3]
        except IndexError:
            model_name, config_name = "", ""

        config_parts = parse_config_name(config_name)
        df["Model"] = model_name
        for k, v in config_parts.items():
            df[k] = v

        all_frames.append(df)

    result = pd.concat(all_frames, ignore_index=True)

    # 🔽 添加排序逻辑：Task ↑, Metric ↓, Value ↑
    return result.sort_values(by=["Task", "Metric", "Value"], ascending=[True, True, False]).reset_index(drop=True)


In [5]:
# ========== 使用 ==========
root = "/mnt/public/code/hanyu/codes/SEAP/eval_out"
df_all = collect_eval_results(root)

# 可选统一格式
pd.options.display.float_format = "{:.4f}".format
df_all

,Task,Version,Filter,Shot,Metric,Arrow,Value,Stderr,Display,Model,calib,prompt,method,strategy,ratio
0,arc_challenge,1.0000,none,0,acc,↑,0.4718,0.0146,0.4718 ± 0.0146,Llama-2-13b-hf,arc_challenge,cot,WIFV,logistic,0.2
1,arc_challenge,1.0000,none,0,acc,↑,0.4710,0.0146,0.4710 ± 0.0146,Llama-2-13b-hf,arc_easy,cot,WIFV,logistic,0.2
2,arc_challenge,1.0000,none,0,acc,↑,0.4710,0.0146,0.4710 ± 0.0146,Llama-2-13b-hf,openbookqa,cot,WIFV,logistic,0.2
3,arc_challenge,1.0000,none,0,acc,↑,0.4676,0.0146,0.4676 ± 0.0146,Llama-2-13b-hf,openbookqa,knowledge,WIFV,logistic,0.2
4,arc_challenge,1.0000,none,0,acc,↑,0.4650,0.0146,0.4650 ± 0.0146,Llama-2-13b-hf,openbookqa,icl,WIFV,logistic,0.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
411,winogrande,1.0000,none,0,acc,↑,0.6875,0.0130,0.6875 ± 0.0130,Llama-2-13b-hf,c4,icl,WIFV,logistic,0.2
412,winogrande,1.0000,none,0,acc,↑,0.6875,0.0130,0.6875 ± 0.0130,Llama-2-13b-hf,winogrande,icl,WIFV,logistic,0.2
413,winogrande,1.0000,none,0,acc,↑,0.6843,0.0131,0.6843 ± 0.0131,Llama-2-13b-hf,piqa,knowledge,WIFV,logistic,0.2
414,winogrande,1.0000,none,0,acc,↑,0.6803,0.0131,0.6803 ± 0.0131,Llama-2-13b-hf,c4,cot,WIFV,logistic,0.2


In [6]:
output_path = "/mnt/public/code/hanyu/codes/SEAP/eval_summary.xlsx"
df_all.to_excel(output_path, index=False)

print(f"Saved: {output_path}")

Saved: /mnt/public/code/hanyu/codes/SEAP/eval_summary.xlsx
